In [1]:
# Libraries

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row


In [2]:
# Spark Session

spark = SparkSession.builder \
    .appName("DAT535-Project-Pluto") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 14:50:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/16 14:50:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
import os

print(os.getcwd())

/home/ubuntu/spark-notebooks/NYC311-Pluto/notebooks


# Load in a DataFrame

In [64]:
PLUTO_PATH = "/home/ubuntu/spark-project-data/bronze/pluto/pluto.csv"

pluto_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PLUTO_PATH)
)


# Check all columns

In [11]:
print(pluto_df.columns)

['borough', 'Tax block', 'Tax lot', 'community board', 'census tract 2010', 'cb2010', 'schooldist', 'council district', 'postcode', 'firecomp', 'policeprct', 'healtharea', 'sanitboro', 'sanitsub', 'address', 'zonedist1', 'zonedist2', 'zonedist3', 'zonedist4', 'overlay1', 'overlay2', 'spdist1', 'spdist2', 'spdist3', 'ltdheight', 'splitzone', 'bldgclass', 'landuse', 'easements', 'ownertype', 'ownername', 'lotarea', 'bldgarea', 'comarea', 'resarea', 'officearea', 'retailarea', 'garagearea', 'strgearea', 'factryarea', 'otherarea', 'areasource', 'numbldgs', 'numfloors', 'unitsres', 'unitstotal', 'lotfront', 'lotdepth', 'bldgfront', 'bldgdepth', 'ext', 'proxcode', 'irrlotcode', 'lottype', 'bsmtcode', 'assessland', 'assesstot', 'exempttot', 'yearbuilt', 'yearalter1', 'yearalter2', 'histdist', 'landmark', 'builtfar', 'residfar', 'commfar', 'facilfar', 'affresfar', 'mnffar', 'borocode', 'BBL', 'condono', 'tract2010', 'xcoord', 'ycoord', 'latitude', 'longitude', 'zonemap', 'zmcode', 'sanborn', '

# Displays the data type

In [63]:
pluto_df.printSchema()

root
 |-- borough: string (nullable = true)
 |-- Tax block: integer (nullable = true)
 |-- Tax lot: integer (nullable = true)
 |-- community board: integer (nullable = true)
 |-- census tract 2010: double (nullable = true)
 |-- cb2010: integer (nullable = true)
 |-- schooldist: integer (nullable = true)
 |-- council district: integer (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- firecomp: string (nullable = true)
 |-- policeprct: integer (nullable = true)
 |-- healtharea: integer (nullable = true)
 |-- sanitboro: integer (nullable = true)
 |-- sanitsub: string (nullable = true)
 |-- address: string (nullable = true)
 |-- zonedist1: string (nullable = true)
 |-- zonedist2: string (nullable = true)
 |-- zonedist3: string (nullable = true)
 |-- zonedist4: string (nullable = true)
 |-- overlay1: string (nullable = true)
 |-- overlay2: string (nullable = true)
 |-- spdist1: string (nullable = true)
 |-- spdist2: string (nullable = true)
 |-- spdist3: string (nullable = tru

In [16]:
pluto_df.select(
    "BBL",
    "borough",
    "address",
    "landuse",
    "bldgclass",
    "unitsres",
    "yearbuilt",
    "bldgarea"
).show(5, truncate=False)

+----------+-------+-------------------------+-------+---------+--------+---------+--------+
|BBL       |borough|address                  |landuse|bldgclass|unitsres|yearbuilt|bldgarea|
+----------+-------+-------------------------+-------+---------+--------+---------+--------+
|4087860042|QN     |84-74 257 STREET         |1      |A1       |1       |1945     |1,864   |
|4087860046|QN     |84-82 257 STREET         |1      |A0       |1       |1945     |960     |
|4087860051|QN     |85-01 256 STREET         |1      |A0       |1       |1945     |1,068   |
|4087980002|QN     |85-59 LITTLE NECK PARKWAY|1      |A0       |1       |1945     |960     |
|4087980003|QN     |85-55 LITTLE NECK PARKWAY|1      |A0       |1       |1945     |960     |
+----------+-------+-------------------------+-------+---------+--------+---------+--------+
only showing top 5 rows



# Explore landuse Categories

In [18]:
pluto_df.groupBy("landuse") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

[Stage 19:==============>                                           (1 + 3) / 4]

+-------+------+
|landuse| count|
+-------+------+
|      1|566810|
|      2|131460|
|      4| 56272|
|     11| 24663|
|      5| 21243|
|      3| 13288|
|      8| 12055|
|      6|  9303|
|     10|  9157|
|      7|  6346|
|      9|  4775|
|   NULL|  2912|
+-------+------+



# Explore Building Classes (bldgclass)

In [19]:
pluto_df.groupBy("bldgclass") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

[Stage 22:===========================================>              (3 + 1) / 4]

+---------+------+
|bldgclass| count|
+---------+------+
|       A1|122813|
|       A5|104550|
|       B1| 85840|
|       B2| 80002|
|       C0| 71525|
|       B3| 61967|
|       A2| 43463|
|       B9| 25885|
|       A9| 18841|
|       V0| 17741|
|       C1| 16717|
|       C3| 15918|
|       S2| 14657|
|       C2| 13574|
|       A3|  8547|
|       A0|  8417|
|       C7|  7722|
|       K1|  7585|
|       K4|  6305|
|       V1|  5988|
+---------+------+
only showing top 20 rows



# Inspect the BBL values

In [29]:
pluto_df.select("BBL").show(20, truncate=False)

+----------+
|BBL       |
+----------+
|4087860042|
|4087860046|
|4087860051|
|4087980002|
|4087980003|
|4087980005|
|4087980007|
|4087980009|
|4087980011|
|4087980013|
|4087980016|
|4087980017|
|4087980019|
|4087980022|
|4087980024|
|4087980026|
|4087980028|
|4087980030|
|4087980032|
|4087980034|
+----------+
only showing top 20 rows



 # Check BBL length

In [30]:
from pyspark.sql import functions as F

pluto_df.groupBy(
    F.length(F.trim(F.col("BBL"))).alias("bbl_length")
).count().orderBy("bbl_length").show()

[Stage 32:=============================>                            (2 + 2) / 4]

+----------+------+
|bbl_length| count|
+----------+------+
|        10|858284|
+----------+------+



# Check whether BBL has a valid format

In [32]:
pluto_df.groupBy(
    F.when(
        F.trim(F.col("BBL")).rlike("^[1-5][0-9]{9}$"),
        "Valid"
    ).otherwise("Invalid").alias("bbl_status")
).count().show()

[Stage 35:=============================>                            (2 + 2) / 4]

+----------+------+
|bbl_status| count|
+----------+------+
|     Valid|858284|
+----------+------+



# Check for Duplicate BBLs

In [33]:
pluto_df.groupBy("BBL") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(10, truncate=False)

+---+-----+
|BBL|count|
+---+-----+
+---+-----+



# Check residential-unit values (unitsres)


In [34]:
pluto_df.select("unitsres").show(20, truncate=False)

+--------+
|unitsres|
+--------+
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
+--------+
only showing top 20 rows



# Explore the distribution of residential units

In [35]:
pluto_df.groupBy("unitsres") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

[Stage 44:=============================>                            (2 + 2) / 4]

+--------+------+
|unitsres|count |
+--------+------+
|1       |322392|
|2       |272686|
|0       |90072 |
|3       |76562 |
|4       |21703 |
|6       |14257 |
|8       |7818  |
|5       |5751  |
|10      |3673  |
|7       |2879  |
|16      |2511  |
|20      |2478  |
|9       |2211  |
|12      |1857  |
|24      |1155  |
|15      |1125  |
|18      |989   |
|11      |902   |
|14      |857   |
|21      |767   |
+--------+------+
only showing top 20 rows



# Check missing residential-unit values

In [36]:
pluto_df.filter(F.col("unitsres").isNull()).count()

649

# Check for empty strings in unitsres

In [37]:
pluto_df.filter(
    F.trim(F.col("unitsres")) == ""
).count()

0

# Check for non-integer formats in unitsres

In [38]:
pluto_df.filter(
    F.col("unitsres").isNotNull() &
    (~F.trim(F.col("unitsres")).rlike(r"^[0-9]+$"))
).count()

92

# Examine the 92 unusual values

In [39]:
pluto_df.filter(
    F.col("unitsres").isNotNull() &
    (~F.trim(F.col("unitsres")).rlike(r"^[0-9]+$"))
).select("unitsres").distinct().show(20, truncate=False)

[Stage 56:>                                                         (0 + 4) / 4]

+--------+
|unitsres|
+--------+
|1,614   |
|1,084   |
|1,328   |
|1,641   |
|1,712   |
|1,023   |
|1,308   |
|1,321   |
|10,914  |
|1,887   |
|1,170   |
|1,050   |
|1,594   |
|1,379   |
|3,378   |
|1,250   |
|1,267   |
|5,860   |
|1,861   |
|1,115   |
+--------+
only showing top 20 rows



# Verify all unusual values are comma-formatted numbers

In [40]:
pluto_df.filter(
    F.col("unitsres").isNotNull() &
    (~F.trim(F.col("unitsres")).rlike(
        r"^([0-9]+|[0-9]{1,3}(,[0-9]{3})+)$"
    ))
).count()

0

# Summary — unitsres Data Quality \
Check	Result \
Total PLUTO records	858,284 \
NULL values	649 \
Zero residential units	90,072 \
Comma-formatted numbers	92 \
Other invalid formats	0

# Inspect yearbuilt

In [41]:
pluto_df.select("yearbuilt").show(20, truncate=False)

+---------+
|yearbuilt|
+---------+
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
|1945     |
+---------+
only showing top 20 rows



# Check the distribution of yearbuilt

In [42]:
pluto_df.groupBy("yearbuilt") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

[Stage 63:>                                                         (0 + 4) / 4]

+---------+-----+
|yearbuilt|count|
+---------+-----+
|1920     |87223|
|1930     |74588|
|1925     |69419|
|1950     |47105|
|1910     |41694|
|0        |39850|
|1940     |37791|
|1960     |36862|
|1931     |30668|
|1955     |26561|
|1935     |25013|
|1945     |24707|
|1901     |22001|
|1899     |20240|
|1965     |19532|
|1970     |17042|
|1915     |15273|
|1975     |9675 |
|1905     |6876 |
|2004     |6517 |
+---------+-----+
only showing top 20 rows



# Check NULL values in yearbuilt

In [43]:
pluto_df.filter(F.col("yearbuilt").isNull()).count()

467

# Check for empty strings in yearbuilt

In [44]:
pluto_df.filter(
    F.trim(F.col("yearbuilt")) == ""
).count()

0

# Check for invalid year formats

In [45]:
pluto_df.filter(
    F.col("yearbuilt").isNotNull() &
    (~F.trim(F.col("yearbuilt")).rlike(r"^(0|[0-9]{4})$"))
).count()

0

# Find the earliest and latest construction years

In [46]:
pluto_df.filter(
    F.col("yearbuilt").isNotNull() &
    (F.col("yearbuilt") != "0")
).agg(
    F.min(F.col("yearbuilt").cast("int")).alias("earliest_year"),
    F.max(F.col("yearbuilt").cast("int")).alias("latest_year")
).show()

[Stage 75:>                                                         (0 + 4) / 4]

+-------------+-----------+
|earliest_year|latest_year|
+-------------+-----------+
|         1652|       2026|
+-------------+-----------+



# Investigate the oldest building records

In [47]:
pluto_df.filter(
    F.col("yearbuilt") == "1652"
).select(
    "BBL",
    "borough",
    "address",
    "bldgclass",
    "landuse",
    "yearbuilt"
).show(10, truncate=False)

[Stage 79:======================================>                   (2 + 1) / 3]

+----------+-------+------------------+---------+-------+---------+
|BBL       |borough|address           |bldgclass|landuse|yearbuilt|
+----------+-------+------------------+---------+-------+---------+
|3079170009|BK     |5902 DITMAS AVENUE|P7       |8      |1652     |
+----------+-------+------------------+---------+-------+---------+



# Examine the oldest construction years

In [48]:
pluto_df.filter(
    F.col("yearbuilt").isNotNull() &
    (F.col("yearbuilt") != "0")
).groupBy("yearbuilt") \
 .count() \
 .orderBy(F.col("yearbuilt").cast("int").asc()) \
 .show(15)

[Stage 80:===========================================>              (3 + 1) / 4]

+---------+-----+
|yearbuilt|count|
+---------+-----+
|     1652|    1|
|     1661|    1|
|     1665|    1|
|     1694|    1|
|     1706|    1|
|     1719|    1|
|     1720|    1|
|     1722|    1|
|     1725|    1|
|     1729|    1|
|     1735|    1|
|     1747|    1|
|     1750|    1|
|     1758|    1|
|     1765|    1|
+---------+-----+
only showing top 15 rows



# Inspect Building Area (bldgarea)  

In [49]:
pluto_df.select("bldgarea").show(20, truncate=False)

+--------+
|bldgarea|
+--------+
|1,864   |
|960     |
|1,068   |
|960     |
|960     |
|2,312   |
|960     |
|2,792   |
|1,792   |
|960     |
|1,532   |
|1,112   |
|2,622   |
|1,602   |
|1,794   |
|1,600   |
|1,112   |
|1,112   |
|1,782   |
|960     |
+--------+
only showing top 20 rows



# Check NULL values in bldgarea  

In [50]:
pluto_df.filter(
    F.col("bldgarea").isNull()
).count()

506

# Check empty strings in bldgarea  

In [51]:
pluto_df.filter(
    F.trim(F.col("bldgarea")) == ""
).count()

0

# Check for invalid building-area formats

In [52]:
pluto_df.filter(
    F.col("bldgarea").isNotNull() &
    (~F.trim(F.col("bldgarea")).rlike(
        r"^([0-9]+|[0-9]{1,3}(,[0-9]{3})+)$"
    ))
).count()

0

# Check buildings with zero area

In [53]:
pluto_df.filter(
    F.regexp_replace(F.col("bldgarea"), ",", "").cast("long") == 0
).count()

40966

Total PLUTO records	858,284 \
Zero building area	40,966 \
Percentage	Approximately 4.77% \
NULL building area	506 \
Invalid numeric formats	0

# Investigate properties with zero building area

In [54]:
pluto_df.filter(
    F.regexp_replace(F.col("bldgarea"), ",", "").cast("long") == 0
).groupBy("landuse") \
 .count() \
 .orderBy("count", ascending=False) \
 .show(15, truncate=False)

[Stage 96:===========================================>              (3 + 1) / 4]

+-------+-----+
|landuse|count|
+-------+-----+
|11     |24653|
|10     |7652 |
|9      |3832 |
|7      |2717 |
|NULL   |1676 |
|1      |377  |
|5      |39   |
|8      |12   |
|4      |3    |
|2      |3    |
|6      |1    |
|3      |1    |
+-------+-----+



# Investigate residential properties with zero building area

In [55]:
pluto_df.filter(
    (F.regexp_replace(F.col("bldgarea"), ",", "").cast("long") == 0) &
    (F.col("landuse").isin("1", "2", "3"))
).select(
    "BBL",
    "landuse",
    "bldgclass",
    "unitsres",
    "yearbuilt",
    "bldgarea"
).show(10, truncate=False)

+----------+-------+---------+--------+---------+--------+
|BBL       |landuse|bldgclass|unitsres|yearbuilt|bldgarea|
+----------+-------+---------+--------+---------+--------+
|4091360232|1      |Z0       |0       |0        |0       |
|3010460057|1      |Z0       |0       |0        |0       |
|3010270027|1      |Z0       |0       |0        |0       |
|3002200021|1      |Z0       |0       |0        |0       |
|3016170034|1      |Z0       |0       |0        |0       |
|3080040030|1      |Z0       |0       |0        |0       |
|4132510019|1      |Z0       |0       |0        |0       |
|4072450017|1      |Z0       |0       |0        |0       |
|5068940003|1      |Z0       |0       |0        |0       |
|3066810321|1      |Z0       |0       |0        |0       |
+----------+-------+---------+--------+---------+--------+
only showing top 10 rows



# Check building classes for residential properties with zero area

In [56]:
pluto_df.filter(
    (F.regexp_replace(F.col("bldgarea"), ",", "").cast("long") == 0) &
    (F.col("landuse").isin("1", "2", "3"))
).groupBy("bldgclass") \
 .count() \
 .orderBy("count", ascending=False) \
 .show(15, truncate=False)

[Stage 100:>                                                        (0 + 4) / 4]

+---------+-----+
|bldgclass|count|
+---------+-----+
|Z0       |371  |
|B2       |3    |
|B1       |1    |
|A5       |1    |
|D1       |1    |
|C0       |1    |
|A1       |1    |
|C1       |1    |
|C2       |1    |
+---------+-----+



#  Investigate the 10 non-Z0 records

In [57]:
pluto_df.filter(
    (F.regexp_replace(F.col("bldgarea"), ",", "").cast("long") == 0) &
    (F.col("landuse").isin("1", "2", "3")) &
    (F.col("bldgclass") != "Z0")
).select(
    "BBL",
    "landuse",
    "bldgclass",
    "unitsres",
    "yearbuilt",
    "bldgarea"
).show(20, truncate=False)

[Stage 104:>                                                        (0 + 3) / 3]

+----------+-------+---------+--------+---------+--------+
|BBL       |landuse|bldgclass|unitsres|yearbuilt|bldgarea|
+----------+-------+---------+--------+---------+--------+
|3021560001|1      |B1       |2       |2025     |0       |
|4131560025|1      |A5       |1       |0        |0       |
|4155350001|3      |D1       |57      |2024     |0       |
|2047180011|2      |C0       |3       |0        |0       |
|4130620060|1      |B2       |2       |2025     |0       |
|4112040047|1      |A1       |1       |1950     |0       |
|3008920052|2      |C2       |6       |1931     |0       |
|3050070067|2      |C1       |9       |0        |0       |
|3010340062|1      |B2       |2       |1899     |0       |
|3034340154|1      |B2       |0       |0        |0       |
+----------+-------+---------+--------+---------+--------+



# Find the maximum building area

In [58]:
pluto_df.select(
    F.regexp_replace(
        F.col("bldgarea"), ",", ""
    ).cast("long").alias("building_area_sqft")
).agg(
    F.max("building_area_sqft").alias("max_building_area_sqft")
).show()

[Stage 105:==============>                                          (1 + 3) / 4]

+----------------------+
|max_building_area_sqft|
+----------------------+
|              24000000|
+----------------------+



#  Investigate the largest building areas

In [59]:
pluto_df.withColumn(
    "building_area_numeric",
    F.regexp_replace(F.col("bldgarea"), ",", "").cast("long")
).orderBy(
    F.col("building_area_numeric").desc()
).select(
    "BBL",
    "borough",
    "address",
    "landuse",
    "bldgclass",
    "numbldgs",
    "building_area_numeric"
).show(10, truncate=False)

[Stage 108:>                                                        (0 + 4) / 4]

+----------+-------+------------------------+-------+---------+--------+---------------------+
|BBL       |borough|address                 |landuse|bldgclass|numbldgs|building_area_numeric|
+----------+-------+------------------------+-------+---------+--------+---------------------+
|3085911500|BK     |FLATBUSH AVENUE         |8      |Y4       |6       |24000000             |
|4142600001|QN     |154-68 BROOKVILLE BLVD  |7      |T1       |223     |17095037             |
|2051410120|BX     |2049 BARTOW AVENUE      |3      |D4       |250     |13540113             |
|1018650001|MN     |1021 AMSTERDAM AVENUE   |8      |M1       |7       |13435076             |
|4163500500|QN     |422 ROCKAWAY POINT BOULE|9      |Q9       |68      |12252000             |
|1000580001|MN     |185 GREENWICH STREET    |5      |O4       |9       |8966500              |
|1009720001|MN     |240 1 AVENUE            |4      |D7       |39      |8942176              |
|3020230001|BK     |652 KENT AVENUE         |7    

Property address	Building area (sq. ft.)	Number of buildings \
Flatbush Avenue, Brooklyn	24,000,000	6 \
154-68 Brookville Blvd, Queens	17,095,037	223 \
2049 Bartow Avenue, Bronx	13,540,113	250 \
1021 Amsterdam Avenue, Manhattan	13,435,076	7 \
422 Rockaway Point Boulevard, Queens	12,252,000	68

#  Investigate the property reporting 24 million sq. ft.

In [60]:
pluto_df.filter(
    F.col("BBL") == "3085911500"
).select(
    "BBL",
    "address",
    "landuse",
    "bldgclass",
    "numbldgs",
    "numfloors",
    "lotarea",
    "bldgarea",
    "resarea",
    "comarea",
    "unitsres",
    "yearbuilt"
).show(truncate=False)

[Stage 110:>                                                        (0 + 3) / 3]

+----------+---------------+-------+---------+--------+---------+---------+----------+-------+----------+--------+---------+
|BBL       |address        |landuse|bldgclass|numbldgs|numfloors|lotarea  |bldgarea  |resarea|comarea   |unitsres|yearbuilt|
+----------+---------------+-------+---------+--------+---------+---------+----------+-------+----------+--------+---------+
|3085911500|FLATBUSH AVENUE|8      |Y4       |6       |NULL     |2,413,870|24,000,000|0      |24,000,000|0       |1930     |
+----------+---------------+-------+---------+--------+---------+---------+----------+-------+----------+--------+---------+



# Count properties with building area above 1 million sq. ft.

In [61]:
pluto_df.filter(
    F.regexp_replace(
        F.col("bldgarea"), ",", ""
    ).cast("long") > 1000000
).count()

268

Building-area check	Result \
Total PLUTO records	858,284 \
Zero building area	40,966 \
Missing building area	506 \
Greater than 1 million sq. ft.	268 \
Invalid numeric formats	0

# Inspecting 3 more columns


In [70]:
pluto_df.select(
    "unitstotal",
    "numbldgs",
    "lotarea"
).show(20, truncate=False)


+----------+--------+-------+
|unitstotal|numbldgs|lotarea|
+----------+--------+-------+
|1         |2       |4,000  |
|1         |1       |3,100  |
|1         |1       |4,000  |
|1         |2       |4,794  |
|1         |2       |4,520  |
|1         |2       |4,760  |
|1         |1       |5,000  |
|1         |2       |5,280  |
|1         |1       |5,520  |
|1         |2       |5,800  |
|1         |2       |4,000  |
|1         |2       |4,200  |
|1         |2       |4,224  |
|1         |1       |4,200  |
|1         |2       |4,000  |
|1         |2       |4,000  |
|1         |2       |4,000  |
|1         |2       |4,000  |
|1         |2       |4,000  |
|1         |2       |4,000  |
+----------+--------+-------+
only showing top 20 rows

